# Ensemble forecast evaluation

This notebook demonstrates evaluation of **ensemble forecasts** using
`ForecastPerformance`.  Topics covered:
* Adding an ensemble simulation
* Fair CRPS (accounts for finite ensemble size)
* Post-processing: mean and scale adjustment (`adjustMean`, `adjustScale`)
* Expected value extraction and deterministic skill
* `leadtimePlot` for multi-leadtime visualisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from performance import ForecastPerformance
from performance import rmse, mae

## 1  Synthetic data

In [ ]:
rng = np.random.default_rng(2)

dates = pd.date_range("2018-01-01", periods=365 * 3, freq="D")
t = np.arange(len(dates))

reference = pd.Series(
    10 + 8 * np.sin(2 * np.pi * t / 365) + rng.normal(0, 1, len(t)),
    index=dates,
    name="Reference",
)

## 2  Build a 20-member ensemble

Each member is the reference plus a slightly biased, noisy perturbation.

In [ ]:
N_MEMBERS = 20
SPREAD = 2.0      # ensemble spread (std)
MEAN_BIAS = 1.5   # systematic mean bias to demonstrate adjustMean

ens_array = np.stack(
    [
        reference.values + MEAN_BIAS + rng.normal(0, SPREAD, len(dates))
        for _ in range(N_MEMBERS)
    ],
    axis=1,
)

ens_df = pd.DataFrame(
    ens_array,
    index=dates,
    columns=pd.MultiIndex.from_product(
        [[pd.Timedelta("0D")], range(N_MEMBERS)],
        names=["Leadtime", "Ensemble member"],
    ),
)

fp = ForecastPerformance(reference)
fp.add_by_production_date(ens_df, name="raw_ens")
print("Simulation type:", fp.simulations["raw_ens"]["simulationType"])

## 3  Fair CRPS

The Fair CRPS applies a bias correction for finite ensemble size.

In [ ]:
lt = pd.Timedelta("0D")

crps_raw = fp.CRPS("raw_ens", leadtime=lt)
fair_raw = fp.fairCRPS("raw_ens", reference="raw_ens", leadtime=lt)
print(f"Raw ensemble CRPS:      {crps_raw:.4f}")
print(f"Fair CRPSS (self-ref):  {fair_raw:.4f}")

## 4  Post-processing: `adjustMean` and `adjustScale`

* `adjustMean` removes the systematic mean bias.
* `adjustScale` corrects the ensemble spread to match the observed variability.

These methods return a new `ForecastPerformance` object with the adjusted
simulation added under a new name.

In [ ]:
fp.adjustMean("raw_ens")
fp.adjustScale("raw_ens_mean_adjusted")

print("Registered simulations after adjustment:", fp.names)

## 5  Skill comparison after post-processing

In [ ]:
rows = []
for name in fp.names:
    ev = fp.get_expected_value(name, leadtime=lt)
    tmp = pd.concat((ev, reference), axis=1).dropna()
    sim_vals = tmp.iloc[:, 0].values
    ref_vals = tmp["Reference"].values
    rows.append({
        "Simulation": name,
        "RMSE": round(rmse(sim_vals, ref_vals), 4),
        "MAE":  round(mae(sim_vals, ref_vals), 4),
        "CRPS": round(fp.CRPS(name, leadtime=lt), 4),
    })

pd.DataFrame(rows).set_index("Simulation")

## 6  Multi-leadtime ensemble

Build a 3-leadtime ensemble where spread grows with lead time, then
visualise CRPS across leadtimes with `leadtimePlot`.

In [ ]:
leadtimes = [pd.Timedelta(f"{d}D") for d in range(4)]
frames = []
for lt_ in leadtimes:
    spread_ = SPREAD * (1 + lt_.days * 0.3)
    members = np.stack(
        [reference.values + rng.normal(0, spread_, len(dates)) for _ in range(N_MEMBERS)],
        axis=1,
    )
    frames.append(
        pd.DataFrame(
            members,
            index=dates,
            columns=pd.MultiIndex.from_product(
                [[lt_], range(N_MEMBERS)],
                names=["Leadtime", "Ensemble member"],
            ),
        )
    )

multi_lt_ens = pd.concat(frames, axis=1)
fp.add_by_production_date(multi_lt_ens, name="multi_lt_ens")
print("Multi-leadtime sim registered.  Leadtimes:", fp.simulations["multi_lt_ens"]["leadtimes"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
fp.leadtimePlot("multi_lt_ens", metric=fp.CRPS, ax=ax)
ax.set_title("CRPS vs Leadtime")
ax.set_ylabel("CRPS")
plt.tight_layout();

## 7  Forecast envelope visualisation

In [ ]:
prod_dates = dates[::180][:4]
fig, ax = plt.subplots(figsize=(12, 4))
fp.plot("raw_ens", production_dates=prod_dates, legend=True, ax=ax)
ax.set_title("Ensemble forecast envelopes")
plt.tight_layout();